# 부스팅(Boosting) — XGBoost 분류 · 회귀

수업 중 작성한 필기 `앙상블분류실습` · `앙상불회귀실습01` 의 XGBoost 부분을 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `머신러닝` p.92~107 — 앙상블 · 부스팅과 XGBoost)

- 원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.
- 그림 · **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 미리 만들어 두었고, 아래 코드를 실행하면 같은 그림이 다시 그려집니다.
- 배깅(랜덤 포레스트)은 [Day 024](../../2026-10-13/) , 스태킹은 [`stacking.ipynb`](./stacking.ipynb) 에 정리했습니다.

> 실습 데이터 `house_price.csv` · `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` 는 이 노트북과 **같은 폴더**에 있어야 합니다.
> XGBoost 설치 → `pip install xgboost`

In [ ]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 앙상블 세 가지 (교재 p.92~93)

| 방식 | 학습 | 대표 모델 | |
|------|------|----------|---|
| **배깅** (Bagging) | 복원 추출한 서브 데이터로 트리를 **따로따로(병렬)** → 투표 / 평균 | RandomForest | [Day 024](../../2026-10-13/) |
| **부스팅** (Boosting) | 이전 모델이 **틀린 만큼(잔차)** 을 다음 모델이 보완 — **순서대로(순차)** | XGBoost | **오늘** |
| **스태킹** (Stacking) | 여러 모델의 **예측값**을 새 특성으로 → **메타 모델**이 최종 예측 | Stacking | [`stacking.ipynb`](./stacking.ipynb) |

## 2. 부스팅의 핵심 — 잔차를 깎아 나가기 (교재 p.102)

**'조각상 다듬기'**

| 단계 | 하는 일 |
|------|---------|
| 초기값 | 큰 덩어리 잡기 — **평균**으로 예측하고 오차(**잔차** = 정답 − 예측) 계산 |
| 1단계 | 튀어나온 큰 오차를 깎는다 — 다음 트리가 **잔차를 예측**, 그 값의 **학습률(예: 0.1)** 만큼만 반영 |
| 2단계 | 덜 깎이거나 잘못 깎인 미세한 오차를 또 깎는다 |
| 반복 | 수백 번 반복 → 점점 정밀한 모델 |

In [ ]:
import numpy as np
import pandas as pd

# 교재 p.102 : 키 · 성별 · 나이로 몸무게 예측
people = pd.DataFrame({
    'Height': [1.8, 1.7, 1.7, 1.5, 1.6],
    'Gender': ['Male', 'Male', 'Male', 'Female', 'Female'],
    'Age': [27, 44, 58, 15, 25],
    'Weight': [88, 68, 76, 35, 54],
})

learning_rate = 0.1

# 초기값 : 평균
pred0 = people['Weight'].mean()
people['잔차(1차)'] = people['Weight'] - pred0

# 1번째 트리가 잔차를 '완벽하게' 맞혔다고 가정하고, 학습률만큼만 반영
pred1 = pred0 + learning_rate * people['잔차(1차)']
people['잔차(2차)'] = (people['Weight'] - pred1).round(2)

print(f"초기 예측(평균) = {pred0}")
people

> 잔차가 **23.8 → 21.42**, **−29.2 → −26.28** 처럼 조금씩(학습률 0.1 만큼씩) 0 으로 다가간다. 트리를 수백 개 쌓으면 잔차가 거의 0 이 된다.

> **📝 교재 오타** — p.102 표에서 2번째 · 3번째 사람의 2차 잔차 식이 `68 − (64.2 + 0.1 × 7.8)`, `76 − (64.2 + 0.1 × 7.8)` 로 적혀 있다.
> 1차 잔차가 각각 **3.8 · 11.8** 이므로 맞는 식은 `68 − (64.2 + 0.1 × 3.8) = 3.42`, `76 − (64.2 + 0.1 × 11.8) = 10.62` 이다.

### 🎬 잔차를 학습하는 트리를 직접 쌓아 보기

`DecisionTreeRegressor(max_depth=2)` 로 **잔차를 학습하는 트리**를 하나씩 더한다. 이게 부스팅(그래디언트 부스팅)의 뼈대다.

![부스팅 잔차 애니메이션](./images/anim_02_boosting_residuals.gif)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# 곡선 모양의 1차원 데이터 60개
rng = np.random.RandomState(0)
x = np.sort(rng.uniform(0, 10, 60))
y = 3 * np.sin(x * 0.8) + 0.4 * x + rng.normal(0, 0.6, 60)
X1 = x.reshape(-1, 1)

LR = 0.3                                   # 학습률
pred = np.full_like(y, y.mean())           # 초기값 : 평균
trees = []

for m in range(1, 51):
    residual = y - pred                                        # ① 잔차 = 정답 - 지금까지의 예측
    tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X1, residual)   # ② 잔차를 학습
    pred = pred + LR * tree.predict(X1)                        # ③ 학습률만큼만 더한다
    trees.append(tree)
    if m in [1, 2, 5, 10, 20, 50]:
        print(f"트리 {m:>2}개 → 학습용 MSE {np.mean((y - pred) ** 2):.3f}")

In [ ]:
# 트리 1개 / 5개 / 50개일 때 예측선 비교
grid = np.linspace(0, 10, 400).reshape(-1, 1)

def boosted_predict(X, k):
    # 평균 + 학습률 × (트리 1 ~ k 의 예측 합)
    return y.mean() + LR * sum(t.predict(X) for t in trees[:k])

plt.figure(figsize=(9, 4))
plt.scatter(x, y, s=15, color='gray', label='학습 데이터')
for k in [1, 5, 50]:
    plt.plot(grid, boosted_predict(grid, k), label=f'트리 {k}개')
plt.legend()
plt.show()

> 트리 하나하나는 깊이 2 짜리 **약한 모델**이지만, 앞이 못 맞힌 부분만 골라 계속 보완하니 50개면 학습용 MSE 가 **0.04** 까지 떨어진다.
> 반대로 말하면 **너무 많이 쌓으면 학습용을 외운다**(과대적합) → 학습률 · 트리 개수 · 깊이로 조절한다.

## 3. XGBoost 사용법 (교재 p.103)

**XGBoost** (eXtreme Gradient Boosting) — 위의 그래디언트 부스팅을 **빠르고 · 규제를 넣어** 구현한 라이브러리.

```python
# pip install xgboost
from xgboost import XGBClassifier          # 분류  ← 교재의 XGBClassifer 는 오타 (i 가 빠짐)
from xgboost import XGBRegressor           # 회귀

xgb = XGBClassifier(random_state=0)
xgb.fit(X_train, y_train)
pred_test_xgb = xgb.predict(X_test)
```

| 주요 매개변수 | 기본값 | 뜻 |
|--------------|-------|-----|
| `n_estimators` | 100 | 트리(부스팅 라운드) 개수 |
| `learning_rate` | 0.3 | 각 트리의 예측을 얼마나 반영할지 (작을수록 천천히 · 신중하게) |
| `max_depth` | 6 | 트리 하나의 최대 깊이 |

## 4. XGBoost 분류 — 당뇨병 여부 (교재 p.104~105)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

df = pd.read_csv('./diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
X_data = df.drop(columns=['Diabetes_binary'])
y_data = df['Diabetes_binary']

# 80 : 20, 정답 비율 유지 (Day 023 · 024 와 같은 분할)
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=0, stratify=y_data
)

# stratify 확인 : 원본 / 학습용 / 평가용 정답 비율
print(pd.DataFrame({'원본': y_data.value_counts(normalize=True),
                    '학습용': y_train.value_counts(normalize=True),
                    '평가용': y_test.value_counts(normalize=True)}).round(3))

In [ ]:
# 모델 생성 · 학습 (트리 100개를 '순서대로' 쌓는다)
xgb = XGBClassifier(random_state=0)
xgb.fit(X_train, y_train)

# 예측 : 평가용 특성(X_test)만 넣는다
pred_test_xgb = xgb.predict(X_test)

acc_xgb = accuracy_score(y_test, pred_test_xgb)
print(f"XGBoost 평가용 정확도 : {acc_xgb:.4f}")
print(f"XGBoost 학습용 정확도 : {accuracy_score(y_train, xgb.predict(X_train)):.4f}")
print("참고 : RandomForest 0.7402 (Day 024) / 단일 결정 트리 0.6545 (Day 023)")

In [ ]:
# 특성 중요도 Top 6
importance = pd.Series(xgb.feature_importances_, index=X_data.columns).sort_values(ascending=False)
print(importance.head(6).round(3))

> - XGBoost **75.0%** — 랜덤 포레스트(74.0%)보다 1%p 높다
> - 학습용 78.6% · 평가용 75.0% — 차이가 작다. 랜덤 포레스트는 학습용 99.5% 였다 (트리를 끝까지 키우는 배깅 vs 얕은 트리를 쌓는 부스팅의 차이)
> - 중요도는 **고혈압(`HighBP`) 0.53** · 전반적 건강(`GenHlth`) 0.12 순 — 단일 트리(Day 023)와 비슷하다. 특성을 고르게 쓰던 랜덤 포레스트(BMI 0.17 · Age 0.13 …)와 달리, 부스팅은 잔차를 가장 잘 줄이는 특성을 반복해서 고른다

## 5. XGBoost 회귀 — 보스턴 집값 (교재 p.106~107)

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error as rmse

df_h = pd.read_csv('./house_price.csv')
X_h = df_h.drop(columns=['medv'])
y_h = df_h['medv']

# random_state 고정 → 다른 모델과 같은 분할로 비교
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_h, test_size=0.25, random_state=0)

# 트리 기반이라 표준화는 필요 없다
xgb_reg = XGBRegressor(random_state=0)
xgb_reg.fit(X_train_h, y_train_h)
pred_test_xgb = xgb_reg.predict(X_test_h)

print(f"XGBRegressor 평가용 RMSE : {rmse(y_test_h, pred_test_xgb):.2f}")
print(f"XGBRegressor 학습용 RMSE : {rmse(y_train_h, xgb_reg.predict(X_train_h)):.2f}")
print("참고 : RandomForest 4.07 (Day 024) / LinearRegression 5.46")

> 평가용 RMSE **4.23** — 랜덤 포레스트(4.07)보다 조금 나쁘다. 그런데 학습용 RMSE 가 **0.01** — 깊이 6 짜리 트리 100개가 학습용 집 379채를 **거의 다 외웠다**.
> 데이터가 작을 때(379채) 기본값은 과대적합되기 쉽다 → **트리를 얕게** 해 본다.

### 🎬 학습률 · 깊이를 바꾸면

![XGBoost 라운드별 RMSE](./images/anim_03_xgb_rounds.gif)

In [ ]:
# 학습률 · 깊이 조합별로 트리 300개까지 쌓으며 평가용 RMSE 기록
#   eval_set 을 주면 라운드마다 성능을 기록한다 (학습에는 쓰이지 않는다)
configs = {
    'lr=0.3, depth=6 (기본값)': dict(learning_rate=0.3, max_depth=6),
    'lr=0.3, depth=3': dict(learning_rate=0.3, max_depth=3),
    'lr=0.05, depth=3': dict(learning_rate=0.05, max_depth=3),
}

plt.figure(figsize=(9, 4))
for name, params in configs.items():
    m = XGBRegressor(n_estimators=300, random_state=0, **params)
    m.fit(X_train_h, y_train_h, eval_set=[(X_test_h, y_test_h)], verbose=False)
    curve = m.evals_result()['validation_0']['rmse']
    plt.plot(range(1, 301), curve, label=f'{name} → 100개 {curve[99]:.2f} / 300개 {curve[-1]:.2f}')
plt.axhline(4.07, color='gray', linestyle='--', label='RandomForest 4.07')
plt.ylim(3, 9)
plt.xlabel('트리 개수')
plt.ylabel('평가용 RMSE')
plt.legend()
plt.show()

In [ ]:
# 깊이만 3 으로 줄인 XGBoost (나머지 기본값)
xgb_d3 = XGBRegressor(max_depth=3, random_state=0).fit(X_train_h, y_train_h)
print(f"XGBRegressor(max_depth=3) 평가용 RMSE : {rmse(y_test_h, xgb_d3.predict(X_test_h)):.2f}")

> - 깊이를 **6 → 3** 으로 줄이기만 해도 RMSE **4.23 → 3.76** — 랜덤 포레스트(4.07)보다 좋아졌다
> - 학습률을 **0.05** 로 줄이면 훨씬 천천히 내려가서(100개에서 4.19) 트리가 더 많이 필요하다
> - 다만 이 조합들을 **평가용 점수를 보고** 고르면 평가용에 맞춰진 셈이다. 실전에서는 교차 검증(`GridSearchCV`)으로 고른다

> **📝 고친 부분** (`앙상블분류실습` · `앙상불회귀실습01`)
> - 회귀 실습의 부스팅 모델이 **`XGBRFRegressor`** 였다 → 이름에 RF 가 들어간 대로 XGBoost 로 만든 **랜덤 포레스트(배깅)** 라 부스팅 실습이 아니다. 교재대로 **`XGBRegressor`** 로 교체
>   (참고 : 같은 분할에서 `XGBRFRegressor` 는 RMSE 3.98. 최신 xgboost 에서는 deprecated 경고가 뜬다)
> - `train_test_split` 에 **`random_state` 가 없어** 필기의 RMSE 2.68 · 2.72 는 다시 만들 수 없는 값 → `random_state=0` 고정
> - 회귀에 쓰지 않는 `accuracy_score` 임포트 제거, `"예상 값:"` 으로 RMSE 를 찍던 문구 정리
> - 교재 `XGBClassifer` 오타 → **`XGBClassifier`**
> - 경로 `C:\Users\…` → `./`, `warnings.filterwarnings('ignore')` 제거
> - 잔차를 직접 학습하는 부스팅 시뮬레이션, 학습률 · 깊이 비교 추가

## 6. 정리

| | 배깅 (RandomForest) | 부스팅 (XGBoost) |
|---|---|---|
| 학습 순서 | 병렬 (트리끼리 독립) | **순차** (앞 트리의 잔차를 다음 트리가) |
| 트리 | 깊게(끝까지) | **얕게** (기본 깊이 6) |
| 핵심 매개변수 | `n_estimators`, `max_features` | `n_estimators`, **`learning_rate`**, `max_depth` |
| 당뇨병 정확도 | 74.0% | **75.0%** |
| 보스턴 RMSE | 4.07 | 4.23 → `max_depth=3` 이면 **3.76** |